# SNN Exercises
### Hands-on explorations to deepen your understanding

These exercises have no single correct answer. The goal is to **experiment, observe, and develop intuition**.

Run the setup cell first, then tackle any exercise in any order.

## Before you start

Activate the `sinabs_tutorial` conda environment as your kernel. See [README.md](README.md) for setup instructions. Complete the N-MNIST tutorial first — exercises here build on those concepts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import sinabs
import sinabs.layers as sl

torch.manual_seed(42)
np.random.seed(42)

# Reusable IAF simulation from the tutorial
def iaf_neuron_sim(inputs, threshold=1.0, leak=1.0):
    T = len(inputs)
    vmem = np.zeros(T + 1)
    spikes = np.zeros(T)
    for t in range(T):
        vmem[t + 1] = leak * vmem[t] + inputs[t]
        if vmem[t + 1] >= threshold:
            spikes[t] = 1
            vmem[t + 1] -= threshold
    return vmem[1:], spikes

print("Setup complete.")

---
## Exercise 1: Predict before you run

Before running any code, look at the input sequence below and **predict**:
- At which timestep(s) will the neuron fire?
- What will Vmem be just before the first spike?
- What will Vmem be at the very last timestep?

```
inputs    = [0.3, 0.4, 0.2, 0.5, 0.1, 0.3, 0.4, 0.2]
threshold = 1.0
```

Work it out on paper first. Then run the cell below to check.

In [ ]:
inputs = [0.3, 0.4, 0.2, 0.5, 0.1, 0.3, 0.4, 0.2]
threshold = 1.0

vmem, spikes = iaf_neuron_sim(inputs, threshold=threshold)

print("t | input | Vmem  | spike")
print("-" * 32)
for t in range(len(inputs)):
    print(f"{t} | {inputs[t]:.1f}   | {vmem[t]:.2f}  | {'SPIKE' if spikes[t] else ''}")

**Now dig deeper:** The reset subtracts exactly `threshold` from Vmem — it does not zero it out. Why is this design choice important? What would happen differently if the reset always set Vmem to zero?

In [ ]:
# Modify iaf_neuron_sim or write your own version with a hard reset (Vmem → 0)
# Compare the spike patterns between the two reset strategies

# YOUR EXPLORATION HERE

---
## Exercise 2: The silent neuron

A neuron that never fires contributes nothing to the network's output and nothing to the loss. It is dead weight.

Your task: construct an input sequence where the neuron accumulates charge for all 20 timesteps but **never quite reaches threshold**. Then answer: what does the surrogate gradient look like for this neuron throughout training? Is it getting any useful gradient signal?

In [ ]:
T = 20
threshold = 1.0

# Design an input sequence that keeps Vmem just below threshold the whole time
inputs = # YOUR DESIGN HERE

vmem, spikes = iaf_neuron_sim(inputs, threshold=threshold)

# Compute surrogate gradient (SingleExponential) at each timestep
beta = 4.0
sg = np.exp(-beta * np.abs(vmem - threshold))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(vmem, marker='o', color='darkorange')
ax1.axhline(threshold, color='red', linestyle='--', label='threshold')
ax1.set_ylabel('Vmem'); ax1.legend(); ax1.set_title('Membrane voltage')
ax2.bar(range(T), sg, color='steelblue')
ax2.set_ylabel('Surrogate gradient'); ax2.set_xlabel('Timestep')
ax2.set_title('Gradient signal available to backprop')
plt.tight_layout(); plt.show()
print(f"Total spikes: {int(spikes.sum())}")

**Follow-up:** Now try the **MultiGaussian** surrogate instead. Its negative lobes are specifically designed to handle this situation. Compute and plot the MultiGaussian gradient for the same silent neuron. What does a negative gradient signal mean for a weight update?

In [ ]:
# MultiGaussian: positive peak at threshold + negative lobes on either side
# Parameters from the tutorial: h=0.5, h2=0.25, mu2=0.5, sigma1=0.3, sigma2=0.3

# YOUR EXPLORATION HERE

---
## Exercise 3: What happens if you forget `reset_states`?

In the training loop, `sinabs.reset_states(model)` is called after each batch to clear Vmem. What happens if you remove it?

There are two distinct effects to understand:

**Effect 1 — Broken backprop (training crashes):**  
Sinabs stores `v_mem` as a tensor with a gradient graph attached. Without `reset_states()`, `v_mem` from batch N is still connected to batch N's computation graph when batch N+1 runs its forward pass. Calling `.backward()` on batch N+1's loss tries to traverse batch N's already-freed graph → immediate `RuntimeError`.  
Run the cell below to see the crash yourself, then read the explanation after.

**Effect 2 — State contamination (wrong inference even without training):**  
Even in `torch.no_grad()` mode (no crash possible), Vmem from sample A leaks into sample B. The second cell below shows this directly.


In [ ]:

import tonic
import tonic.transforms as transforms
from torch.utils.data import DataLoader
from tonic.collation import PadTensors

BATCH_SIZE = 32
NUM_TIMESTEPS = 10
SENSOR_SIZE = tonic.datasets.NMNIST.sensor_size

frame_transform = transforms.Compose([
    transforms.Denoise(filter_time=10_000),
    transforms.ToFrame(sensor_size=SENSOR_SIZE, n_time_bins=NUM_TIMESTEPS),
])
dataset = tonic.datasets.NMNIST(save_to="./data", train=True, transform=frame_transform)
dataset = tonic.DiskCachedDataset(dataset, cache_path="./data/cache/nmnist_train")
loader = DataLoader(
    torch.utils.data.Subset(dataset, range(3200)),
    batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    collate_fn=PadTensors(batch_first=True),
)


def build_snn(batch_size, num_timesteps):
    return nn.Sequential(
        nn.Conv2d(2, 16, kernel_size=3, padding=1, bias=False),
        sl.IAFSqueeze(batch_size=batch_size, num_timesteps=num_timesteps),
        nn.AvgPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False),
        sl.IAFSqueeze(batch_size=batch_size, num_timesteps=num_timesteps),
        nn.AvgPool2d(2),
        nn.Flatten(),
        nn.Linear(32 * 8 * 8, 10, bias=False),
        sl.IAFSqueeze(batch_size=batch_size, num_timesteps=num_timesteps),
    )


# ── Effect 1: Crash demo ─────────────────────────────────────────────────────
print("=== Effect 1: missing reset_states() crashes training ===\n")
model_broken = build_snn(BATCH_SIZE, NUM_TIMESTEPS)
optimizer = torch.optim.Adam(model_broken.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

try:
    for batch_idx, (x, y) in enumerate(loader):
        x = x.float().reshape(BATCH_SIZE * NUM_TIMESTEPS, 2, 34, 34)
        spike_counts = model_broken(x).reshape(BATCH_SIZE, NUM_TIMESTEPS, -1).sum(dim=1)
        loss = loss_fn(spike_counts, y.long())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # ← reset_states() is missing here
        if batch_idx == 1:   # crash happens on the 2nd backward pass
            break
    print("No crash — unexpected!")
except RuntimeError as e:
    print(f"RuntimeError on batch 2:\n  {str(e)[:120]}...\n")
    print("Why: v_mem from batch 1 is still attached to batch 1's freed graph.")
    print("When batch 2 calls .backward(), PyTorch tries to traverse that freed graph.")
    print("Fix: call sinabs.reset_states(model) after every optimizer.step().")


In [ ]:
# ── Effect 2: State contamination — same input, different output ──────────────
# No model or dataset needed — iaf_neuron_sim is enough to show the effect.

T = 15
threshold = 1.0

# Sample A: a sequence that leaves significant residual Vmem when it ends
inputs_A = [0.12] * T
vmem_A, spikes_A = iaf_neuron_sim(inputs_A, threshold=threshold)
residual = vmem_A[-1]
print(f"Sample A ends with Vmem = {residual:.2f}  ← residual charge left in the neuron")

# Sample B: identical input in both cases — the only difference is starting Vmem
inputs_B = [0.07] * T

def iaf_from_vmem(inputs, threshold=1.0, v0=0.0):
    """Same as iaf_neuron_sim but with a configurable starting Vmem."""
    T = len(inputs)
    vmem = np.zeros(T + 1)
    vmem[0] = v0
    spikes = np.zeros(T)
    for t in range(T):
        vmem[t + 1] = vmem[t] + inputs[t]
        if vmem[t + 1] >= threshold:
            spikes[t] = 1
            vmem[t + 1] -= threshold
    return vmem[1:], spikes

vmem_clean,        spikes_clean        = iaf_from_vmem(inputs_B, v0=0.0)
vmem_contaminated, spikes_contaminated = iaf_from_vmem(inputs_B, v0=residual)

print(f"\nSample B — clean start (Vmem=0):          "
      f"{int(spikes_clean.sum())} spike(s) at t={list(np.where(spikes_clean > 0)[0])}")
print(f"Sample B — contaminated start (Vmem={residual:.2f}): "
      f"{int(spikes_contaminated.sum())} spike(s) at t={list(np.where(spikes_contaminated > 0)[0])}")
print("\nSame input. Different output. Because reset_states() was skipped.")

ts = np.arange(T)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
fig.suptitle("Effect 2: same input (Sample B), different spike timing — because of Sample A's leftover Vmem",
             fontsize=12, fontweight='bold')

for ax, vmem, spikes, title, color, v0 in zip(
        axes,
        [vmem_clean, vmem_contaminated],
        [spikes_clean, spikes_contaminated],
        [f'With reset_states()  →  Vmem starts at 0',
         f'Without reset_states()  →  Vmem starts at {residual:.2f}'],
        ['steelblue', 'darkorange'],
        [0.0, residual]):
    spike_times = np.where(spikes > 0)[0]
    vmem_disp = vmem.copy()
    for t in spike_times:
        vmem_disp[t] = vmem[t] + threshold   # show pre-reset height for arrow

    ax.plot(ts, vmem_disp, color=color, linewidth=2, marker='o', markersize=4)
    ax.axhline(threshold, color='red', linestyle='--', linewidth=1.5, label='Threshold')
    ax.axhline(v0, color='gray', linestyle=':', linewidth=1.2,
               label=f'Starting Vmem = {v0:.2f}')
    for t in spike_times:
        ax.annotate('', xy=(t, vmem[t]), xytext=(t, vmem_disp[t]),
                    arrowprops=dict(arrowstyle='->', color='red', lw=2))
        ax.text(t, vmem_disp[t] + 0.06, 'SPIKE', ha='center',
                fontsize=8, color='red', fontweight='bold')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Timestep'); ax.set_ylabel('Vmem')
    ax.set_ylim(-0.05, 1.55); ax.set_xticks(ts)
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
    ax.text(0.98, 0.96, f'Spikes: {int(spikes.sum())}',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=10, color='red', fontweight='bold')

plt.tight_layout()
plt.show()


**Think about it:** When Vmem is not reset between batches, what exactly is leaking from one sample into the next? Why does this cause the loss to behave the way you observed?

---
## Exercise 4: The leaky neuron (LIF)

The IAF neuron we used has perfect memory — charge accumulated at t=0 is still there at t=63. Real biological neurons leak: if no input arrives, the membrane slowly returns to rest.

The **Leaky Integrate-and-Fire (LIF)** neuron adds a decay factor `leak` (0 < leak < 1):

```
Vmem[t+1] = leak × Vmem[t] + input[t]
```

The `iaf_neuron_sim` in the setup cell already supports this via the `leak` parameter.

Experiment with `leak = 1.0, 0.9, 0.7, 0.5` on the same input sequence. Plot all four Vmem traces on the same axes.

In [ ]:
T = 20
np.random.seed(0)
inputs = np.random.uniform(0.05, 0.2, T)  # weak, noisy input
threshold = 1.0
leaks = [1.0, 0.9, 0.7, 0.5]

# YOUR EXPLORATION HERE
# Plot Vmem traces for each leak value
# Count total spikes for each
# At what leak value does the neuron stop firing altogether?

**Think about it:** A high leak value makes the neuron more forgetful — it responds to recent inputs but ignores distant ones. How would you expect this to affect performance on a dataset where the useful events are spread across many timesteps vs concentrated in a few?

---
## Exercise 5: Surrogate gradient width and training

The `beta` parameter of the SingleExponential surrogate controls how sharply it peaks:

- **Large beta** → narrow peak → only neurons very close to threshold receive gradient
- **Small beta** → wide peak → neurons further from threshold also receive gradient

sinabs exposes this: `sinabs.activation.SingleExponential(grad_width=...)` (where `grad_width = 1/beta`).

First, visualise the surrogate shape for several beta values to build intuition.

In [ ]:
vmem_vals = np.linspace(-1.0, 3.0, 500)
threshold = 1.0
betas = [1.0, 4.0, 10.0, 25.0]

# YOUR EXPLORATION HERE
# Plot the surrogate gradient shape for each beta on the same axes
# Label which beta is which

Now train two small models — one with a narrow surrogate (high beta) and one with a wide surrogate (low beta) — on the same 3000-sample subset. Compare their loss curves.

Use `sinabs.activation.SingleExponential(grad_width=1/beta)` and pass it as `spike_fn` to `IAFSqueeze`.

In [ ]:
# Hint: IAFSqueeze accepts a spike_fn argument
# sl.IAFSqueeze(batch_size=B, num_timesteps=T,
#               spike_fn=sinabs.activation.SingleExponential(grad_width=...))

# YOUR EXPLORATION HERE

---
## Exercise 6: How many timesteps do you actually need?

More timesteps give the network more time to accumulate evidence — but also means slower training (the IAF loop runs longer). There is a tradeoff.

Train the same model architecture with `NUM_TIMESTEPS = 1, 5, 10, 20`. For each, train for 2 epochs on the full dataset and record test accuracy.

In [ ]:
# Note: you need to rebuild the dataset with a new frame_transform for each T value
# because ToFrame(n_time_bins=T) changes the output shape
# Also rebuild the DataLoader and model each time

timestep_options = [1, 5, 10, 20]
results = {}  # {T: test_accuracy}

# YOUR EXPLORATION HERE

# After collecting results, plot accuracy vs NUM_TIMESTEPS
# Does accuracy keep improving as T grows, or does it plateau?

**Think about it:** At `T=1` there is no temporal integration at all — the neuron either fires or not based on a single input. In that case, how is the SNN different from a standard ANN with a threshold activation?

---
## Exercise 7: Spike counts as confidence

After training, the network's "confidence" for a class is just how many times the corresponding output neuron fired. Take 20 test samples and for each one plot the spike count of all 10 output neurons as a bar chart.

Find at least one sample where the network is **wrong** — the highest spike count belongs to the wrong class. Look at the input frames for that sample. Can you see why the network was confused?

In [ ]:
# Load the trained model from the tutorial notebook, or train a fresh one here
# Then run inference on 20 test samples

# For each sample:
#   - bar chart of spike counts per class
#   - mark the true label in green, the predicted label in red if wrong

# YOUR EXPLORATION HERE

---
## Exercise 8: Replace spikes with ReLU — does it still work?

What if we replaced every `IAFSqueeze` layer with a standard `nn.ReLU`? The network becomes a regular ANN — no spikes, no Vmem, no surrogate gradients needed.

Train both versions (SNN and ANN-with-ReLU) on the same data for the same number of epochs. Compare their accuracy curves.

In [ ]:
def build_ann(batch_size, num_timesteps):
    # Same architecture as the SNN, but IAFSqueeze replaced with ReLU
    # Note: ReLU doesn't need batch_size or num_timesteps
    # The squeeze (B*T) still happens — ReLU just processes a big flat batch like Conv2d does
    return nn.Sequential(
        nn.Conv2d(2, 16, kernel_size=3, padding=1, bias=False),
        nn.ReLU(),
        nn.AvgPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False),
        nn.ReLU(),
        nn.AvgPool2d(2),
        nn.Flatten(),
        nn.Linear(32 * 8 * 8, 10, bias=False),
        nn.ReLU(),
    )

# YOUR EXPLORATION HERE
# Train both models and compare
# Does the ANN need the spike-count accumulation step in the loss?
# (Hint: ReLU outputs are continuous, not binary spikes — what should you use as logits?)

**Think about it:** If the ANN achieves similar or better accuracy, why would anyone use an SNN? What does the ANN lose that the SNN preserves?